# SQL GROUPING SETS : plusieurs agrégations en un seul scan

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/moncoachdata/MonShotData/blob/main/SQL/Grouping-Sets-SQL.ipynb)

📖 **Le shot complet** : [SQL GROUPING SETS : la technique sous-estimée pour agréger en un scan](https://www.monshotdata.com/p/sql-grouping-sets-agregations-multiples-scan-unique)

---

Pour obtenir plusieurs agrégations dans une même table de résultats, le réflexe est d'écrire une requête par agrégation puis de les assembler avec `UNION ALL`. Ça marche, mais la table est **parcourue autant de fois que de requêtes**. `GROUPING SETS` fait tout en une seule requête, avec un seul parcours.

**À la fin de ce notebook, tu sauras :**

- écrire plusieurs agrégations (par ville, puis par ville et statut) avec `UNION ALL`, puis avec `GROUPING SETS` ;
- vérifier que les deux écritures donnent le même résultat ;
- voir dans le plan d'exécution que `UNION ALL` parcourt la table deux fois et `GROUPING SETS` une seule, et mesurer l'écart sur une grande table.

### Avant de lancer

- **Installation** : la première cellule installe **DuckDB**, un moteur SQL qui tourne dans le notebook.
- **Les données** : un fichier CSV de 1 000 employés fictifs (`employees_grouping_sets.csv`, dossier `_datasets` du dépôt). Il est téléchargé automatiquement depuis GitHub s'il n'est pas déjà à côté du notebook.
- **La grande table** : à la section 6, on génère 20 millions de lignes directement dans DuckDB pour mesurer le temps d'exécution (quelques secondes, environ 300 Mo de mémoire). Les durées dépendent de ta machine : regarde surtout l'écart entre les deux requêtes.

🧪 *Testé le 22 novembre 2026 : exécution complète de haut en bas (Python 3.13, DuckDB 2.0, pandas 3.0). Les sorties affichées viennent de cette exécution.*

## 1. Mise en place

In [1]:
%pip install -q duckdb

In [2]:
import os
import re
import time
import urllib.request

import duckdb
import pandas as pd

# Télécharge le jeu de données depuis le dépôt GitHub s'il n'est pas déjà présent
FICHIER = "employees_grouping_sets.csv"
URL = "https://raw.githubusercontent.com/moncoachdata/MonShotData/main/_datasets/" + FICHIER
if not os.path.exists(FICHIER):
    urllib.request.urlretrieve(URL, FICHIER)

## 2. Le jeu de données

Une table `employees` : un employé par ligne, avec sa ville (`City`) et son statut (`Status`, temps plein ou stagiaire).

In [3]:
employees = pd.read_csv(FICHIER)
employees.head()

,Name,City,Country,Salary,Status,Rating
0,Johnny Maynard,New Cindychester,Mongolia,764340,Full Time,3.3
1,Michael Williams,Ricardomouth,Guinea-Bissau,929230,Full Time,4.6
2,Laura Flynn,North Melissafurt,Tonga,606900,Intern,0.3
3,Stefanie Archer,Ricardomouth,San Marino,775380,Full Time,2.8
4,Sierra Garcia,Whitakerbury,Nepal,944480,Full Time,4.5


In [4]:
print(employees.shape)
print(employees["Status"].value_counts().to_dict())
print(employees["City"].nunique(), "villes")

(1000, 6)
{'Full Time': 791, 'Intern': 209}
10 villes


On veut, **dans une seule table** :

- le nombre total d'employés par `City` (agrégation sur `City`) ;
- le nombre d'employés à temps plein et de stagiaires par `City` (agrégation sur `City` **et** `Status`).

Chaque ville apparaît donc sur trois lignes : le total, le temps plein et les stagiaires. Pour la ligne « total », la colonne `Status` vaut `NULL`.

## 3. L'approche classique : `UNION ALL`

Une requête par agrégation, assemblées par `UNION ALL`. On calcule le total en mettant `NULL` dans la colonne `Status`, pour que les deux requêtes aient les mêmes colonnes.

In [5]:
query_union = """
SELECT City, NULL AS Status, COUNT(*) AS TotalEmployees
FROM employees
GROUP BY City

UNION ALL

SELECT City, Status, COUNT(*) AS TotalEmployees
FROM employees
GROUP BY City, Status

ORDER BY City, Status;
"""

resultat_union = duckdb.sql(query_union).df()
resultat_union.head(6)

,City,Status,TotalEmployees
0,Aliciafort,Full Time,89
1,Aliciafort,Intern,24
2,Aliciafort,NaN,113
3,Kristaburgh,Full Time,77
4,Kristaburgh,Intern,20
5,Kristaburgh,NaN,97


Ça fonctionne. Mais chaque `SELECT` lit la table `employees` en entier : **deux parcours complets** de la même table.

## 4. L'approche intelligente : `GROUPING SETS`

On déclare toutes les agrégations voulues dans une seule clause `GROUP BY GROUPING SETS (...)` : ici `(City)` et `(City, Status)`. Les colonnes qui ne font pas partie de l'ensemble de regroupement valent `NULL`.

In [6]:
query_gs = """
SELECT City, Status, COUNT(*) AS TotalEmployees
FROM employees
GROUP BY GROUPING SETS (
    (City),
    (City, Status)
)
ORDER BY City, Status;
"""

resultat_gs = duckdb.sql(query_gs).df()
resultat_gs.head(6)

,City,Status,TotalEmployees
0,Aliciafort,Full Time,89
1,Aliciafort,Intern,24
2,Aliciafort,NaN,113
3,Kristaburgh,Full Time,77
4,Kristaburgh,Intern,20
5,Kristaburgh,NaN,97


## 5. Le résultat est-il identique ?

In [7]:
# Mêmes lignes, dans le même ordre
resultat_union.equals(resultat_gs)

True

`True` : les deux écritures produisent exactement la même table. Mais `GROUPING SETS` est plus concise, et la table n'est parcourue qu'**une seule fois**, quel que soit le nombre d'agrégations demandées.

Un détail utile : dans le résultat, un `Status` à `NULL` peut signifier « total de la ville » **ou** une vraie valeur manquante dans les données. La fonction `GROUPING()` lève le doute : elle vaut 1 quand la colonne est agrégée (la ligne de total) et 0 sinon.

In [8]:
query_grouping = """
SELECT City, Status, GROUPING(Status) AS ligne_de_total, COUNT(*) AS TotalEmployees
FROM employees
GROUP BY GROUPING SETS ((City), (City, Status))
ORDER BY City, Status;
"""

duckdb.sql(query_grouping).df().head(6)

,City,Status,ligne_de_total,TotalEmployees
0,Aliciafort,Full Time,0,89
1,Aliciafort,Intern,0,24
2,Aliciafort,NaN,1,113
3,Kristaburgh,Full Time,0,77
4,Kristaburgh,Intern,0,20
5,Kristaburgh,NaN,1,97


## 6. Un parcours contre deux : le plan d'exécution

`EXPLAIN` affiche le plan d'exécution de DuckDB. On compte combien de fois la table `employees` y est parcourue (`PANDAS_SCAN`, le nom du parcours d'un DataFrame Pandas).

In [9]:
def nombre_de_scans(requete):
    plan = duckdb.sql("EXPLAIN " + requete).fetchall()[0][1]
    return len(re.findall("PANDAS_SCAN", plan))

pd.DataFrame({
    "Approche": ["UNION ALL", "GROUPING SETS"],
    "Parcours de la table": [nombre_de_scans(query_union), nombre_de_scans(query_gs)],
})

,Approche,Parcours de la table
0,UNION ALL,2
1,GROUPING SETS,1


### Et sur une grande table ?

Sur 1 000 lignes, la différence de temps est invisible. On génère donc une table de **20 millions de lignes** (1 000 villes, deux statuts) dans DuckDB, puis on mesure les deux requêtes (meilleur temps de 3 essais).

In [10]:
duckdb.sql("""
CREATE OR REPLACE TABLE gros AS
SELECT
    (random() * 1000)::INT AS City,
    CASE WHEN random() < 0.5 THEN 'Full Time' ELSE 'Intern' END AS Status
FROM range(20000000)
""")

print(duckdb.sql("SELECT COUNT(*) AS lignes FROM gros").fetchall()[0][0], "lignes")

20000000 lignes


In [11]:
def meilleur_temps(requete, essais=3):
    temps = []
    for _ in range(essais):
        debut = time.perf_counter()
        duckdb.sql(requete).fetchall()
        temps.append(time.perf_counter() - debut)
    return min(temps)

q_union = query_union.replace("FROM employees", "FROM gros").replace("ORDER BY City, Status", "")
q_gs = query_gs.replace("FROM employees", "FROM gros").replace("ORDER BY City, Status", "")

t_union = meilleur_temps(q_union)
t_gs = meilleur_temps(q_gs)

pd.DataFrame({
    "Approche": ["UNION ALL", "GROUPING SETS"],
    "Temps (s)": [round(t_union, 3), round(t_gs, 3)],
})

,Approche,Temps (s)
0,UNION ALL,0.383
1,GROUPING SETS,0.325


In [12]:
print(f"GROUPING SETS est {t_union / t_gs:.2f}× plus rapide que UNION ALL sur cette machine")

GROUPING SETS est 1.18× plus rapide que UNION ALL sur cette machine


L'écart de temps reste modeste sur ce cas : DuckDB est très rapide pour parcourir une table, et le travail d'agrégation compte autant que la lecture. Il grandit avec le coût de la lecture (grosse table sur disque ou dans le cloud, nombreuses colonnes, requêtes facturées au volume lu) et avec le nombre d'agrégations demandées : avec cinq agrégations, `UNION ALL` lit la table cinq fois, `GROUPING SETS` une fois.

## 7. Pour aller plus loin : `ROLLUP` et `CUBE`

`GROUPING SETS` est la forme générale. `ROLLUP` et `CUBE` en sont des raccourcis pour les cas fréquents. `ROLLUP (City, Status)` équivaut à `GROUPING SETS ((City, Status), (City), ())` : le détail, le sous-total par ville et le total général.

In [13]:
duckdb.sql("""
SELECT City, Status, COUNT(*) AS TotalEmployees
FROM employees
GROUP BY ROLLUP (City, Status)
ORDER BY City NULLS LAST, Status NULLS LAST
""").df().tail(6)

,City,Status,TotalEmployees
25,Whitakerbury,Intern,21
26,Whitakerbury,NaN,87
27,Whiteside,Full Time,85
28,Whiteside,Intern,19
29,Whiteside,NaN,104
30,NaN,NaN,1000


La dernière ligne, avec `City` et `Status` à `NULL`, est le total général.

## 8. Ce qu'il faut retenir

- Pour plusieurs agrégations dans une même table, `UNION ALL` relit la table autant de fois qu'il y a de requêtes.
- `GROUP BY GROUPING SETS ((A), (A, B))` produit les mêmes lignes en **un seul parcours** de la table.
- Les colonnes absentes d'un ensemble de regroupement valent `NULL` dans les lignes correspondantes ; `GROUPING(col)` permet de les distinguer d'une vraie valeur manquante.
- Le gain de temps dépend du coût de lecture de la table et du nombre d'agrégations : modeste en mémoire, plus net sur de gros volumes ou des données lues à distance.
- `ROLLUP` et `CUBE` sont des raccourcis de `GROUPING SETS` pour les sous-totaux hiérarchiques et toutes les combinaisons.

---

## Pour aller plus loin

- [NOT IN en SQL : l'erreur invisible qui te coûte des heures](https://www.monshotdata.com/p/erreur-silencieuse-sql-not-in-null) — un autre piège SQL classique
- [Semi, Anti et Natural Join : 3 joins SQL méconnus](https://www.monshotdata.com/p/semi-join-anti-join-natural-join-sql) — les joins que tu connais mal
- [Tous les notebooks des shots](https://github.com/moncoachdata/MonShotData) — le code Python de la newsletter, rangé par thème